In [1]:
!pip install pretty_midi tqdm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.6/5.6 MB 47.5 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 54.6/54.6 kB 4.3 MB/s eta 0:00:00
  Created wheel for pretty_midi: filename=pretty_midi-0.2.11-py3-none-any.whl size=5595886 sha256=3a8b3d70ae259f2174b7aa0ce4d68473f2a4975e4f10f94f8312287937fe0080
  Stored in directory: /root/.cache/pip/wheels/f4/ad/93/a7042fe12668827574927ade9deec7f29aad2a1001b1501882
Successfully built pretty_midi


# Load Libraries

In [2]:
import os
import glob
import math
import random
import numpy as np
from tqdm import tqdm
import pretty_midi

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

In [3]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

cuda


In [4]:
DATA_PATH = "/kaggle/input/lakh-midi-clean/"

midi_files = glob.glob(DATA_PATH + "/**/*.mid", recursive=True)
print("Total MIDI files:", len(midi_files))

midi_files = midi_files[:500]
print("Using:", len(midi_files))

Total MIDI files: 17219
Using: 500


# Converting song to Notes

In [5]:
# Simplify MIDI instrument programs into coarse categories

def simplify_program(program):
    if program <= 7:
        return 0  # Piano
    elif program <= 15:
        return 1  # Chromatic Percussion
    elif program <= 23:
        return 2  # Organ
    elif program <= 31:
        return 3  # Guitar
    elif program <= 39:
        return 4  # Bass
    elif program <= 47:
        return 5  # Strings
    elif program <= 55:
        return 6  # Ensemble
    elif program <= 63:
        return 7  # Brass
    elif program <= 71:
        return 8  # Reed
    elif program <= 79:
        return 9  # Pipe
    else:
        return 10  # Other
        


In [6]:
def midi_to_tokens(midi_path, time_step=0.02, max_shift=50):
    midi = pretty_midi.PrettyMIDI(midi_path)
    events = []

    for instrument in midi.instruments:
        program = simplify_program(instrument.program)

        for note in instrument.notes:
            if 21 <= note.pitch <= 108:
                events.append((note.start, f"INST_{program}_NOTE_ON_{note.pitch}"))
                events.append((note.end, f"INST_{program}_NOTE_OFF_{note.pitch}"))

    events.sort(key=lambda x: x[0])

    tokens = []
    prev_time = 0.0

    # 🔥 Add START token
    tokens.append("START")

    for event_time, token in events:
        delta = event_time - prev_time
        steps = int(delta / time_step)

        while steps > 0:
            shift = min(steps, max_shift)
            tokens.append(f"TIME_SHIFT_{shift}")
            steps -= shift

        tokens.append(token)
        prev_time = event_time

    # 🔥 Add END token
    tokens.append("END")

    return tokens

In [7]:
all_tokens = []

for path in tqdm(midi_files):
    try:
        tokens = midi_to_tokens(path)
        all_tokens.extend(tokens)
    except:
        continue

vocab = sorted(list(set(all_tokens)))
token_to_idx = {t: i for i, t in enumerate(vocab)}
idx_to_token = {i: t for t, i in token_to_idx.items()}

vocab_size = len(vocab)

print("New Vocabulary Size:", vocab_size)

  1%|          | 3/500 [00:00<01:33,  5.32it/s]/usr/local/lib/python3.12/dist-packages/pretty_midi/pretty_midi.py:122: RuntimeWarning: Tempo, Key or Time signature change events found on non-zero tracks.  This is not a valid type 0 or type 1 MIDI file.  Tempo, Key or Time Signature may be wrong.
  warnings.warn(
100%|██████████| 500/500 [01:21<00:00,  6.10it/s]


New Vocabulary Size: 1712


# Convert Notes to Numerics

In [8]:
SEQ_LEN = 128
all_sequences = []

for path in tqdm(midi_files):
    try:
        tokens = midi_to_tokens(path)
        encoded = [token_to_idx[t] for t in tokens if t in token_to_idx]

        for i in range(len(encoded) - SEQ_LEN):
            x = encoded[i:i+SEQ_LEN]
            y = encoded[i+1:i+SEQ_LEN+1]
            all_sequences.append((x, y))

    except:
        continue

print("Total sequences:", len(all_sequences))

100%|██████████| 500/500 [02:41<00:00,  3.09it/s]

Total sequences: 6492318


# Test Train Split

In [9]:
import random

random.shuffle(all_sequences)

split = int(0.9 * len(all_sequences))
train_sequences = all_sequences[:split]
val_sequences = all_sequences[split:]

train_sequences = random.sample(train_sequences, 800000)
val_sequences = random.sample(val_sequences, 100000)

print("Train:", len(train_sequences))
print("Val:", len(val_sequences))

Train: 800000
Val: 100000


# DATASET LOADER

In [10]:
class MusicDataset(Dataset):
    def __init__(self, sequences):
        self.data = sequences

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        x, y = self.data[idx]
        return (
            torch.tensor(x, dtype=torch.long),
            torch.tensor(y, dtype=torch.long)
        )

In [11]:
train_loader = DataLoader(
    MusicDataset(train_sequences),
    batch_size=32,
    shuffle=True,
    num_workers=2,
    pin_memory=True
)

val_loader = DataLoader(
    MusicDataset(val_sequences),
    batch_size=32,
    num_workers=2,
    pin_memory=True
)

print("Train batches:", len(train_loader))
print("Val batches:", len(val_loader))

Train batches: 25000
Val batches: 3125


In [12]:
import math

class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=5000):
        super().__init__()

        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len).unsqueeze(1)

        div_term = torch.exp(
            torch.arange(0, d_model, 2) *
            (-math.log(10000.0) / d_model)
        )

        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)

        pe = pe.unsqueeze(0)
        self.register_buffer("pe", pe)

    def forward(self, x):
        return x + self.pe[:, :x.size(1)]

# Mask to hide future tokens

In [13]:
def generate_causal_mask(seq_len):
    mask = torch.triu(torch.ones(seq_len, seq_len), diagonal=1)
    mask = mask.masked_fill(mask == 1, float("-inf"))
    return mask

In [14]:
class MusicTransformer(nn.Module):
    def __init__(self, vocab_size, d_model=256, nhead=8, num_layers=4):
        super().__init__()

        self.embedding = nn.Embedding(vocab_size, d_model)
        self.pos_encoder = PositionalEncoding(d_model)

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=nhead,
            dim_feedforward=512,
            batch_first=True
        )

        self.transformer = nn.TransformerEncoder(
            encoder_layer,
            num_layers=num_layers
        )

        self.fc = nn.Linear(d_model, vocab_size)

    def forward(self, x):
        x = self.embedding(x)
        x = self.pos_encoder(x)

        mask = generate_causal_mask(x.size(1)).to(x.device)

        x = self.transformer(x, mask)
        x = self.fc(x)

        return x

In [15]:
model = MusicTransformer(vocab_size).to(device)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.0003)

print("Model initialized.")

Model initialized.


# Training

In [16]:
EPOCHS = 2

for epoch in range(EPOCHS):

    model.train()
    train_loss = 0.0

    for x, y in train_loader:
        x = x.to(device)
        y = y.to(device)

        optimizer.zero_grad()

        output = model(x)

        loss = criterion(
            output.reshape(-1, vocab_size),
            y.reshape(-1)
        )

        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()

        train_loss += loss.item()

    train_loss /= len(train_loader)

    model.eval()
    val_loss = 0.0

    with torch.no_grad():
        for x, y in val_loader:
            x = x.to(device)
            y = y.to(device)

            output = model(x)

            loss = criterion(
                output.reshape(-1, vocab_size),
                y.reshape(-1)
            )

            val_loss += loss.item()

    val_loss /= len(val_loader)

    print("Train Loss:", train_loss)
    print("Val Loss:", val_loss)
    print("Val PPL:", torch.exp(torch.tensor(val_loss)))

Train Loss: 2.655518201560974
Val Loss: 2.033950115356445
Val PPL: tensor(7.6442)
Train Loss: 2.185434177927971
Val Loss: 1.8615978288269044
Val PPL: tensor(6.4340)


# Generate Music

In [17]:
def generate_music_from_start(model, max_length=1200, temperature=0.85, top_k=8):
    model.eval()

    start_token = token_to_idx["START"]
    end_token = token_to_idx["END"]

    generated = [start_token]

    with torch.no_grad():
        for _ in range(max_length):

            x = torch.tensor(
                generated[-SEQ_LEN:]
            ).unsqueeze(0).to(device)

            output = model(x)
            logits = output[:, -1, :] / temperature
            probs = torch.softmax(logits, dim=-1)

            top_probs, top_idx = torch.topk(probs, top_k)
            top_probs = top_probs / torch.sum(top_probs)

            next_token = torch.multinomial(top_probs, 1)
            next_token = top_idx[0][next_token].item()

            generated.append(next_token)

            # Stop if END token appears
            if next_token == end_token:
                break

    return generated

In [18]:
def tokens_to_midi(tokens, filename="transformer_output.mid", time_step=0.02):
    midi = pretty_midi.PrettyMIDI()
    instruments = {}
    active_notes = {}
    current_time = 0.0

    for idx in tokens:
        token = idx_to_token[idx]

        if token.startswith("TIME_SHIFT"):
            steps = int(token.split("_")[-1])
            current_time += steps * time_step

        elif token.startswith("INST"):
            parts = token.split("_")

            program = int(parts[1])
            action = parts[2] + "_" + parts[3]   # NOTE_ON or NOTE_OFF
            pitch = int(parts[4])

            if program not in instruments:
                instruments[program] = pretty_midi.Instrument(program=program)

            if action == "NOTE_ON":
                active_notes[(program, pitch)] = current_time

            elif action == "NOTE_OFF":
                key = (program, pitch)
                if key in active_notes:
                    note = pretty_midi.Note(
                        velocity=80,
                        pitch=pitch,
                        start=active_notes[key],
                        end=current_time
                    )
                    instruments[program].notes.append(note)
                    del active_notes[key]

    for inst in instruments.values():
        midi.instruments.append(inst)

    midi.write(filename)
    print("Saved:", filename)

# Saving it in MID Format

In [19]:
generated_tokens = generate_music_from_start(model,2400,1.1,20)
tokens_to_midi(generated_tokens, "transformer_structured.mid")

Saved: transformer_structured.mid
